# Unsupervised learning on advanced diffusion MRI metrics for clinical characterization of migraine.

This script implements the  pipeline for one diffusion-metric set (DTI, AMURA, or DTI+AMURA). The three sets are analyzed with identical code, changing only the input feature matrix.

**Pipeline steps:**

1. Load diffusion metrics and build the feature matrix.
2. Z-score normalization.
3. Dimensionality reduction with an autoencoder.
4. Feature selection based on reconstruction error (median +/- 3*IQR outlier exclusion; the 20 features with the highest reconstruction error are retained).
5. PCA on the selected features.
6. Clustering with K-Means, Agglomerative and Spectral clustering. 
7. The exploratory longitudinal association (CM dataset) is in longitudinal_association.ipynb.

## Setup: reproducibility settings and imports

In [ ]:
import os

# Reproducibility 
os.environ["PYTHONHASHSEED"] = "0"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"

import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from scipy import stats

SEED = 42


def set_seed(seed=SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)

## 1. Preprocessing

In [1]:
def zscore_normalize(df):
       """Z-score each numeric column."""
       numeric = df.select_dtypes(include=["number"])
       return numeric.apply(stats.zscore)

## 2. Autoencoder

In [ ]:
def train_autoencoder(X, latent_dim, epochs=50, batch_size=256):
    X = np.asarray(X, dtype=np.float32)
    n_train = int(0.8 * X.shape[0])
    X_train, X_val = X[:n_train], X[n_train:]

    inp = Input(shape=(X.shape[1],))
    encoded = Dense(latent_dim, activation="relu")(inp)
    decoded = Dense(X.shape[1], activation="linear")(encoded)
    autoencoder = Model(inp, decoded)
    autoencoder.compile(optimizer="adam", loss="mse")

    es = EarlyStopping(patience=3, restore_best_weights=True, monitor="val_loss")
    autoencoder.fit(X_train, X_train, validation_data=(X_val, X_val),
                    epochs=epochs, batch_size=batch_size, shuffle=False,
                    verbose=0, callbacks=[es])
    return autoencoder

## 3. Feature selection by reconstruction error

In [ ]:
def select_features_by_reconstruction_error(df, autoencoder, n_select=20):
    
    X = df.values.astype(np.float32)
    decoded = autoencoder.predict(X, verbose=0)
    rec_error = np.mean((X - decoded) ** 2, axis=0)

    median = np.median(rec_error)
    q1, q3 = np.percentile(rec_error, [25, 75])
    iqr = q3 - q1
    lower, upper = median - 3 * iqr, median + 3 * iqr

    order = np.argsort(rec_error)[::-1]       
    kept = [i for i in order if lower <= rec_error[i] <= upper]
    if len(kept) > n_select:
        kept = kept[:n_select]
    elif len(kept) < 5:
        kept = order[:5]
    return df.iloc[:, kept]

## 4. Clustering

In [ ]:
def make_clusterer(method, k, n_samples):
    if method == "kmeans":
        return KMeans(n_clusters=k, random_state=SEED, n_init=50)
    if method == "agglomerative":
        return AgglomerativeClustering(n_clusters=k)
    if method == "spectral":
        n_neighbors = max(2, min(10, n_samples - 1))
        return SpectralClustering(n_clusters=k, random_state=SEED, n_init=50,
                                  affinity="nearest_neighbors",
                                  n_neighbors=n_neighbors,
                                  assign_labels="kmeans", eigen_solver="arpack")
    raise ValueError(f"Unknown method: {method}")


def run_clustering_pipeline(df_raw, k_best, latent_dim=20):
    set_seed()
    X = zscore_normalize(df_raw)
    autoencoder = train_autoencoder(X.values, latent_dim)
    selected = select_features_by_reconstruction_error(X, autoencoder)
    n_comp = min(latent_dim, selected.shape[1])
    pca_feats = PCA(n_components=n_comp, svd_solver="full",
                    random_state=SEED).fit_transform(selected)

    results = {}
    for method in ("kmeans", "agglomerative", "spectral"):
        labels = make_clusterer(method, k_best, len(pca_feats)).fit_predict(pca_feats)
        results[method] = {
            # labels indexed by patient ID, so they can be matched by ID later
            "labels": pd.Series(labels, index=df_raw.index, name="cluster"),
            "k_best": k_best,
            "silhouette": silhouette_score(pca_feats, labels),
        }
    return results, pca_feats

## Example usage

Patient data are not included for privacy reasons. Set the paths below to the local data files. `FEATURES_PATH` must contain one row per subject (index column `ID`) and one column per diffusion metric and region (one of the DTI / AMURA / DTI+AMURA feature sets). `CLINICAL_PATH` must contain the clinical variables for the same subjects (`;`-separated, column `ID`). The longitudinal analysis is performed in the CM group.

In [ ]:
# Run the pipeline on one of the nine datasets:
# GROUP: "AllPatients", "CM" or "EM"; MODEL: "DTI", "AMURA" or "DTI+AMURA"
GROUP = "CM"
MODEL = "DTI"

FEATURES_PATH = "path/to/features.csv"   # user-provided, features of the chosen GROUP and MODEL
CLINICAL_PATH = "path/to/clinical.csv"   # user-provided

# Optimal number of clusters, determined separately in R with nbclust_optimal_k.R
K_BEST = 2

df_features = pd.read_csv(FEATURES_PATH, index_col="ID")
clin_df = pd.read_csv(CLINICAL_PATH, sep=";", index_col="ID")
clin_df["topiramate_binary"] = clin_df["TopiramateResponse"].map(
    {"Good": 1, "Excellent": 1, "No": 0})

results, pca_feats = run_clustering_pipeline(df_features, k_best=K_BEST, latent_dim=20)
for method, r in results.items():
    print(method, "k =", r["k_best"], "silhouette =", round(r["silhouette"], 3))

# Save PCA features: input for nbclust_optimal_k.R
os.makedirs("results", exist_ok=True)
pd.DataFrame(pca_feats).to_csv(f"results/pca_{GROUP}_{MODEL}.csv", index=False)